In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import torch
from torch import nn
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from tqdm.auto import tqdm

device = (
  "cuda" if torch.cuda.is_available()
  else "mps" if torch.mps.is_available()
  else "cpu"
)

RANDOM_SEED = 42

device

In [ ]:
df_class = pd.read_csv("synthetic_data.csv")
df_class

In [ ]:
df_reg = df_class.copy()

dt = 1e-4

df_reg['displacement'] = df_reg['vibration_rms'] * dt
df_reg['frequency'] = df_reg['spindle_speed_rpm'] / 60
df_reg['velocity'] = 2 * np.pi * df_reg['frequency'] * df_reg['displacement']

k = 2.07e6
c = 2400

df_reg['counter_force'] = k * df_reg['displacement'] + c * df_reg['velocity']

df_reg.drop(columns="label_chatter")

In [ ]:
cols =[
    'vibration_rms',
    'spindle_speed_rpm',
    'depth_of_cut_mm',
    'feed_rate_mm_per_tooth',
    'displacement',
    'velocity',
    'counter_force'
]

df_reg = df_reg[cols]

df_reg

In [ ]:
def convert_to_tensors(
    df: pd.DataFrame, 
    test_split:int,
    device=device,
    scale_y:bool = False
) -> torch.tensor:
  X = df[df.columns[:-1]]
  y = df[df.columns[-1]]

  scaler = StandardScaler()
  X = scaler.fit_transform(X)

  if scale_y:
    y_scaler = StandardScaler()
    y = y.to_numpy().reshape(-1,1)
    y = y_scaler.fit_transform(y)

    X_train, X_test, y_train, y_test = train_test_split(X, y,test_size=test_split)

    X_train, y_train = (
        torch.from_numpy(X_train).float().to(device),
        torch.from_numpy(y_train).float().to(device).squeeze()
    )

    X_test, y_test = (
        torch.from_numpy(X_test).float().to(device),
        torch.from_numpy(y_test).float().to(device).squeeze()
    )

    return X_train, X_test, y_train, y_test

  X_train, X_test, y_train, y_test = train_test_split(X, y,test_size=test_split)

  X_train, y_train = (torch.from_numpy(X_train).type(torch.float).to(device), 
                    torch.from_numpy(y_train.to_numpy()).type(torch.float).to(device).squeeze())

  X_test, y_test = (torch.from_numpy(X_test).type(torch.float).to(device), 
                    torch.from_numpy(y_test.to_numpy()).type(torch.float).to(device).squeeze())
  
  return X_train, X_test, y_train, y_test

In [ ]:
df_reg[df_reg.columns[-1]].to_numpy().reshape(-1, 1).shape

In [ ]:
class ChatterClassification(nn.Module):
  def __init__(self, input_size, hidden_units, output_size):
    super().__init__()
    self.layer_stack = nn.Sequential(
      nn.Linear(in_features=input_size, out_features=hidden_units),
      nn.ReLU(),
      nn.Linear(in_features=hidden_units, out_features=hidden_units // 2),
      nn.ReLU(),
      nn.Linear(in_features=hidden_units // 2, out_features=output_size),
    )

  def forward(self, x):
    return self.layer_stack(x)
  
torch.manual_seed(RANDOM_SEED)

chatter_classification_model = ChatterClassification(
  input_size=5,
  hidden_units=32,
  output_size=1
).to(device)

loss_fn = nn.BCEWithLogitsLoss()
optimizer = torch.optim.SGD(
  params=chatter_classification_model.parameters(),
  lr=0.01
)

chatter_classification_model

In [ ]:
epochs = []
train_losses = []
test_losses = []
accuracy_values = []
test_acc_values = []

X_train, X_test, y_train, y_test = convert_to_tensors(df_class, test_split=0.2)

EPOCHS = 150

for epoch in tqdm(range(EPOCHS), desc="TRAINING...."):
  chatter_classification_model.train()
  y_logits = chatter_classification_model(X_train).squeeze()
  y_pred_prob = torch.sigmoid(y_logits)
  y_pred = (y_pred_prob > 0.5).float()
  loss = loss_fn(y_logits, y_train)
  accuracy = (y_pred == y_train).sum().item() / len(y_pred)
  optimizer.zero_grad()
  loss.backward()
  optimizer.step()

  chatter_classification_model.eval()
  with torch.inference_mode():
    test_logits = chatter_classification_model(X_test).squeeze()
    test_pred_prob = torch.sigmoid(test_logits)
    test_pred = (test_pred_prob > 0.5).float()
    test_loss = loss_fn(test_logits, y_test)
    test_acc = (test_pred == y_test).sum().item() / len(test_pred)

  if epoch % 10 == 0:
    epochs.append(epoch)
    train_losses.append(loss.item())
    test_losses.append(test_loss.item())
    accuracy_values.append(accuracy)
    test_acc_values.append(test_acc)

  if epoch % 30 == 0:
    print(f"EPOCH: {epoch} | LOSS: {loss:.5f}, ACC: {accuracy * 100:.2f}% | TEST LOSS: {test_loss:.5f}, TEST_ACC: {test_acc * 100:.2f}%")

In [ ]:
(y_pred == y_train).sum().item()

In [ ]:
plt.style.use("dark_background")

fig, axs = plt.subplots(1, 2, figsize=(14,4))

axs[0].plot(epochs, train_losses, label="Train Loss", marker="o", markersize=2)
axs[0].plot(epochs, accuracy_values, label="Train Accuracy", marker="o", markersize=2)
axs[0].set_title("Training Metrics")
axs[0].legend()

axs[1].plot(epochs, test_losses, label="Test Loss", marker="o", markersize=2)
axs[1].plot(epochs, test_acc_values, label="Test Accuracy", marker="o", markersize=2)
axs[1].set_title("Test Metrics")
axs[1].legend()

axs[0].set_xlabel("Epoch")
axs[0].set_ylabel("Value")

axs[1].set_xlabel("Epoch")
axs[1].set_ylabel("Value")

for ax in axs:
    ax.grid(True, alpha=0.2)

plt.show()

In [ ]:
import matplotlib

matplotlib.style.available

In [ ]:
chatter_classification_model.eval()
with torch.inference_mode():
  y_preds = (torch.sigmoid(chatter_classification_model(X_test).squeeze()) > 0.5).float()

y_preds.device, y_test.device

In [ ]:
from torchmetrics import ConfusionMatrix
from mlxtend.plotting import plot_confusion_matrix

confusion_mat = ConfusionMatrix(task="binary")

confusion_mat_tensor = confusion_mat(preds=y_preds.cpu(), target=y_test.cpu())

confusion_mat_tensor

In [ ]:
plt.style.use("seaborn-v0_8-darkgrid")

fig, ax = plot_confusion_matrix(
  conf_mat=confusion_mat_tensor.numpy(),
  class_names=["No_Chatter", "Chatter"],
  figsize=(3,3)
)

In [ ]:
from torchinfo import summary

summary = summary(
  model=chatter_classification_model,
  input_size=(400, 5)
)

summary

In [ ]:
print(len(df_reg.columns))

df_reg

In [ ]:
class ForceRegressionModel(nn.Module):
  def __init__(self, input_size, hidden_units, output_size):
    super().__init__()
    self.input_size = input_size
    self.hidden_units = hidden_units
    self.output_size = output_size

    self.layer_stack = nn.Sequential(
      nn.Linear(in_features=input_size, out_features=hidden_units),
      nn.ReLU(),
      nn.Linear(in_features=hidden_units, out_features=hidden_units // 2),
      nn.ReLU(),
      nn.Linear(in_features=hidden_units // 2, out_features=output_size)
    )

  def forward(self, X):
    return self.layer_stack(X)
  
torch.manual_seed(RANDOM_SEED)  

counter_force_model = ForceRegressionModel(
  input_size=6,
  hidden_units=64,
  output_size=1
).to(device)

loss_fn = nn.MSELoss()
optimizer = torch.optim.SGD(
  params=counter_force_model.parameters(),
  lr=1e-4
)

counter_force_model

In [ ]:
epochs = []
train_losses = []
test_losses = []

X_train, X_test, y_train, y_test = convert_to_tensors(df_reg, test_split=0.2, scale_y=True)

EPOCHS = 1000

for epoch in tqdm(range(EPOCHS), desc="TRAINING...."):
  counter_force_model.train()
  y_pred = counter_force_model(X_train).squeeze()
  loss = loss_fn(y_pred, y_train)
  optimizer.zero_grad()
  loss.backward()
  optimizer.step()

  counter_force_model.eval()
  with torch.inference_mode():
    test_pred = counter_force_model(X_test).squeeze()
    test_loss = loss_fn(test_pred, y_test)

  if epoch % 30 == 0:
    epochs.append(epoch)
    train_losses.append(loss.item())
    test_losses.append(test_loss.item())
    accuracy_values.append(accuracy)
    test_acc_values.append(test_acc)
    
  if epoch % 100 == 0:
    print(f"EPOCH: {epoch} | LOSS: {loss:.5f} | TEST LOSS: {test_loss:.5f}")

In [ ]:
plt.style.use("dark_background")

fig, axs = plt.subplots(1, 2, figsize=(14,4))

axs[0].plot(epochs, train_losses, label="Train Loss", c="lightgreen", marker="o", markersize=2)
axs[0].set_title("Training Metrics")
axs[0].legend()

axs[1].plot(epochs, test_losses, label="Test Loss", c="lightblue",marker="o", markersize=2)
axs[1].set_title("Test Metrics")
axs[1].legend()

axs[0].set_xlabel("Epoch")
axs[0].set_ylabel("Value")

axs[1].set_xlabel("Epoch")
axs[1].set_ylabel("Value")

for ax in axs:
    ax.grid(True, alpha=0.2)

plt.show()

In [ ]:
from torchinfo import summary

summary = summary(
  model=counter_force_model,
  input_size=(400, 6)
)

summary

In [ ]:
df_rf = df_reg.copy()

dt = 0.001 

k = 1000     # stiffness
c = 10       # damping

Ki = 10      # actuator current stiffness
Kq = 5       # displacement stiffness

if 'displacement' not in df_rf.columns:
  df_rf['displacement'] = df_rf['vibration_rms'] * dt

if 'velocity' not in df_rf.columns:
  df_rf['velocity'] = 2 * np.pi * df_rf['frequency'] * df_rf['displacement']

if 'force' not in df_rf.columns:
  df_rf['force'] = k * df_rf['displacement'] + c * df_rf['velocity']

df_rf['current'] = (df_rf['force'] - Kq * df_rf['displacement']) / Ki

df_rf = df_rf.replace([np.inf, -np.inf], np.nan)
df_rf = df_rf.dropna()

print(df_rf[['force', 'displacement', 'current']].head())
df_rf

In [ ]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error

X_rf = df_rf[['vibration_rms', 'displacement', 'velocity', 'force',
              'spindle_speed_rpm', 'depth_of_cut_mm', 'feed_rate_mm_per_tooth']].values

y_rf = df_rf['current'].values

X_train, X_test, y_train, y_test = train_test_split(X_rf, y_rf, test_size=0.2, random_state=42)

rf = RandomForestRegressor(n_estimators=100, max_depth=10, random_state=42)

rf.fit(X_train, y_train)

y_pred = rf.predict(X_test)

mse = mean_squared_error(y_test, y_pred)
mae = mean_absolute_error(y_test, y_pred)

print("MSE:", mse)
print("MAE:", mae)

plt.figure(figsize=(8,6))
plt.scatter(y_test, y_pred, c="#dbc548", marker="*", s=16)
plt.xlabel("Actual Current")
plt.ylabel("Predicted Current")
plt.title("RF Current Prediction")
plt.grid(True,alpha=0.2)
plt.show()

feature_names = [
    'vibration_rms',
    'displacement',
    'velocity',
    'force',
    'spindle_speed',
    'depth_of_cut',
    'feed_rate'
]

importances = rf.feature_importances_

plt.figure(figsize=(8, 4))
plt.bar(range(len(importances)), importances, color="#42b0f5")
plt.xticks(range(len(importances)), feature_names, rotation=45)
plt.title("Feature Importance")
plt.tight_layout()
plt.grid(True, alpha=0.2)
plt.show()